In [16]:
import polars as pl
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
import os
from sklearn.preprocessing import StandardScaler

In [17]:


print(f"Polars versión: {pl.__version__}")
# Configuramos Polars para que muestre más columnas cuando hagamos print
pl.Config.set_tbl_cols(20)

Polars versión: 1.35.2


polars.config.Config

In [18]:
import polars as pl
from pathlib import Path

# ==========================================
# 2. CARGA DE DATOS (INYECCIÓN DOBLE)
# ==========================================
carreras = ["australia_2026", "china_2026", "japan_2026", "united_states_2026"]

# CORRECCIÓN AQUÍ: Envolver el string en Path()
BASE_DIR = Path('C:/Users/marce/F1-data-project/project/')

# 1. Carga de Master Parquets CON INYECCIÓN de 'race_name'
master_dfs = []
for c in carreras:
    p = BASE_DIR / "data" / "processed" / f"{c}_masterv2.parquet"
    if p.exists():
        # Inyectamos el nombre de la carrera a la telemetría masiva
        df_m = pl.read_parquet(p).with_columns(pl.lit(c).alias("race_name"))
        master_dfs.append(df_m)
df_master = pl.concat(master_dfs)

# 2. Carga de Eventos CON INYECCIÓN de 'race_name'
event_dfs = []
for c in carreras:
    p = BASE_DIR / "data" / "events" / f"{c}_events.parquet"
    if p.exists():
        df_ev = pl.read_parquet(p).with_columns(pl.lit(c).alias("race_name"))
        event_dfs.append(df_ev)
df_events = pl.concat(event_dfs)

print(f"✅ Telemetría total cargada: {df_master.height} filas")
print(f"✅ Eventos tácticos a procesar: {df_events.height} eventos")

✅ Telemetría total cargada: 2315 filas
✅ Eventos tácticos a procesar: 643 eventos


In [19]:
df_aus = pl.read_parquet('C:\\Users\\marce\\F1-data-project\\project\\data\\events\\australia_2026_events.parquet')

In [20]:
df_aus.columns

['race_id',
 'lap_number',
 'event_type',
 'initiator_driver',
 'target_driver',
 'initiator_compound',
 'initiator_pos_change']

In [21]:
df_master.columns

['meeting_key',
 'session_key',
 'driver_number',
 'lap_number',
 'date_start',
 'duration_sector_1',
 'duration_sector_2',
 'duration_sector_3',
 'i1_speed',
 'i2_speed',
 'is_pit_out_lap',
 'lap_duration',
 'segments_sector_1',
 'segments_sector_2',
 'segments_sector_3',
 'st_speed',
 'position',
 'compound',
 'stint_number',
 'tyre_age',
 'pit_duration',
 'is_pit_lap',
 'throttle_mean_lap',
 'throttle_std_lap',
 'brake_max_lap',
 'rpm_max_lap',
 'n_gear_max_lap',
 'drs_max_lap',
 'race_name']

In [22]:
df_events.columns

['race_id',
 'lap_number',
 'event_type',
 'initiator_driver',
 'target_driver',
 'initiator_compound',
 'initiator_pos_change',
 'race_name']

In [23]:
# ==========================================
# 3. PARÁMETROS, SENSORES Y LIMPIEZA DE TIPOS
# ==========================================
WINDOW_SIZE = 3 

SENSOR_COLS = [
    'lap_duration', 'duration_sector_1', 'duration_sector_2', 'duration_sector_3', 
    'i1_speed', 'i2_speed', 'st_speed', 
    'throttle_mean_lap', 'throttle_std_lap', 
    'brake_max_lap', 'rpm_max_lap', 'n_gear_max_lap', 'drs_max_lap'
]

# 🛠️ LA CORRECCIÓN: Forzamos a que todos los sensores sean números decimales (Float64)
# strict=False hace que si hay un texto inválido (ej. ""), se vuelva Nulo en vez de romper el código
columnas_a_convertir = [col for col in SENSOR_COLS if col in df_master.columns]
df_master = df_master.with_columns([
    pl.col(col).cast(pl.Float64, strict=False) for col in columnas_a_convertir
])

print(f"✅ Telemetría V2 cargada y convertida a números: {df_master.height} filas")
print(f"✅ Eventos a analizar: {df_events.height} eventos")

✅ Telemetría V2 cargada y convertida a números: 2315 filas
✅ Eventos a analizar: 643 eventos


In [24]:
# ==========================================
# 4. FUNCIÓN CORE (VERSIÓN SIN PÉRDIDA DE DATOS)
# ==========================================
def extract_tactical_features(event, df_master):
    race = event.get('race_name', event.get('race_id')) 
    lap = event['lap_number']
    att = event['initiator_driver']
    def_dr = event['target_driver']
    
    # Contexto Base (SIEMPRE se conserva para no perder los 234 eventos)
    feat = {
        'event_id': f"{race}_L{lap}_{att}v{def_dr}",
        'race_name': race,
        'lap_number': lap,
        'event_type': event.get('event_type'),
        'attacker': att,
        'defender': def_dr,
        'pos_change': event.get('initiator_pos_change') 
    }
    
    target_laps = [lap - 1, lap - 2, lap - 3]
    df_race = df_master.filter(pl.col("race_name") == race)
    
    att_data = df_race.filter((pl.col("driver_number") == att) & (pl.col("lap_number").is_in(target_laps)))
    def_data = df_race.filter((pl.col("driver_number") == def_dr) & (pl.col("lap_number").is_in(target_laps)))
    
    # --- BLOQUE DE LLANTAS ---
    att_last_lap = att_data.filter(pl.col("lap_number") == (lap - 1))
    def_last_lap = def_data.filter(pl.col("lap_number") == (lap - 1))
    
    if not att_last_lap.is_empty() and not def_last_lap.is_empty():
        if "compound" in att_last_lap.columns:
            feat['att_compound'] = att_last_lap.select("compound").item()
            feat['def_compound'] = def_last_lap.select("compound").item()
        
        if "tyre_age" in att_last_lap.columns:
            a_tyre = att_last_lap.select(pl.col("tyre_age").cast(pl.Float64, strict=False)).item()
            d_tyre = def_last_lap.select(pl.col("tyre_age").cast(pl.Float64, strict=False)).item()
            if a_tyre is not None and d_tyre is not None:
                feat['delta_tyre_age'] = a_tyre - d_tyre

    # --- BLOQUE MATEMÁTICO EXPANDIDO (5 STATS x 13 SENSORES = 195 COLS) ---
    for sensor in SENSOR_COLS:
        if sensor not in df_master.columns:
            continue
            
        # Al usar .item() en un DataFrame vacío en Polars, devuelve None automáticamente.
        # Esto evita que el código se rompa y asegura que no perdamos la fila del evento.
        
        # 1. Atacante (5 stats)
        feat[f'att_{sensor}_mean'] = att_data.select(pl.col(sensor).mean()).item()
        feat[f'att_{sensor}_max'] = att_data.select(pl.col(sensor).max()).item()
        feat[f'att_{sensor}_min'] = att_data.select(pl.col(sensor).min()).item()
        feat[f'att_{sensor}_std'] = att_data.select(pl.col(sensor).std()).item()
        feat[f'att_{sensor}_median'] = att_data.select(pl.col(sensor).median()).item()
        
        # 2. Defensor (5 stats)
        feat[f'def_{sensor}_mean'] = def_data.select(pl.col(sensor).mean()).item()
        feat[f'def_{sensor}_max'] = def_data.select(pl.col(sensor).max()).item()
        feat[f'def_{sensor}_min'] = def_data.select(pl.col(sensor).min()).item()
        feat[f'def_{sensor}_std'] = def_data.select(pl.col(sensor).std()).item()
        feat[f'def_{sensor}_median'] = def_data.select(pl.col(sensor).median()).item()
        
        # 3. Deltas (Cruciales para el PCA)
        for stat in ['mean', 'max', 'min', 'std', 'median']:
            a_val = feat[f'att_{sensor}_{stat}']
            d_val = feat[f'def_{sensor}_{stat}']
            
            if a_val is not None and d_val is not None:
                feat[f'delta_{sensor}_{stat}'] = a_val - d_val
            else:
                feat[f'delta_{sensor}_{stat}'] = None
            
    return feat

In [25]:
# ==========================================
# 5. EJECUCIÓN Y EXPORTACIÓN
# ==========================================
print("\n⚙️ Calculando 200+ variables tácticas avanzadas... (Recuperando 100% de los eventos)")

features_list = []
for event_row in df_events.to_dicts():
    # Ya no eliminamos eventos, todos pasan
    res = extract_tactical_features(event_row, df_master)
    features_list.append(res)

df_tactical_v2 = pl.DataFrame(features_list)

output_dir = BASE_DIR / "data" / "processed"
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / "tactical_events_v2_FINAL.parquet"

df_tactical_v2.write_parquet(output_path, compression="snappy")

print("\n🚀 ¡PIPELINE CORREGIDO Y COMPLETADO!")
print(f"📊 Total de eventos rescatados: {df_tactical_v2.height} (Debe decir 234)")
print(f"🧬 Variables (Columnas) generadas: {len(df_tactical_v2.columns)} (Debe superar las 200)")
print(f"💾 Guardado en: {output_path.name}")


⚙️ Calculando 200+ variables tácticas avanzadas... (Recuperando 100% de los eventos)

🚀 ¡PIPELINE CORREGIDO Y COMPLETADO!
📊 Total de eventos rescatados: 643 (Debe decir 234)
🧬 Variables (Columnas) generadas: 205 (Debe superar las 200)
💾 Guardado en: tactical_events_v2_FINAL.parquet


In [26]:
print(df_events.columns)

['race_id', 'lap_number', 'event_type', 'initiator_driver', 'target_driver', 'initiator_compound', 'initiator_pos_change', 'race_name']


In [27]:
df_2 = pl.read_parquet('C:\\Users\\marce\\F1-data-project\\project\\data\\processed\\tactical_events_v2_FINAL.parquet')

In [28]:
df_2.shape

(643, 205)

In [29]:
df_2.columns

['event_id',
 'race_name',
 'lap_number',
 'event_type',
 'attacker',
 'defender',
 'pos_change',
 'att_compound',
 'def_compound',
 'delta_tyre_age',
 'att_lap_duration_mean',
 'att_lap_duration_max',
 'att_lap_duration_min',
 'att_lap_duration_std',
 'att_lap_duration_median',
 'def_lap_duration_mean',
 'def_lap_duration_max',
 'def_lap_duration_min',
 'def_lap_duration_std',
 'def_lap_duration_median',
 'delta_lap_duration_mean',
 'delta_lap_duration_max',
 'delta_lap_duration_min',
 'delta_lap_duration_std',
 'delta_lap_duration_median',
 'att_duration_sector_1_mean',
 'att_duration_sector_1_max',
 'att_duration_sector_1_min',
 'att_duration_sector_1_std',
 'att_duration_sector_1_median',
 'def_duration_sector_1_mean',
 'def_duration_sector_1_max',
 'def_duration_sector_1_min',
 'def_duration_sector_1_std',
 'def_duration_sector_1_median',
 'delta_duration_sector_1_mean',
 'delta_duration_sector_1_max',
 'delta_duration_sector_1_min',
 'delta_duration_sector_1_std',
 'delta_duratio

In [30]:
df_2.head()

event_id,race_name,lap_number,event_type,attacker,defender,pos_change,att_compound,def_compound,delta_tyre_age,…,def_drs_max_lap_mean,def_drs_max_lap_max,def_drs_max_lap_min,def_drs_max_lap_std,def_drs_max_lap_median,delta_drs_max_lap_mean,delta_drs_max_lap_max,delta_drs_max_lap_min,delta_drs_max_lap_std,delta_drs_max_lap_median
str,str,i64,str,i64,i64,str,str,str,f64,…,null,null,null,null,null,null,null,null,null,null
"""australia_2026_L4_1v10""","""australia_2026""",4,"""On_Track_Overtake""",1,10,"""P10 -> P7""","""MEDIUM""","""UNKNOWN""",0.0,…,null,null,null,null,null,null,null,null,null,null
"""australia_2026_L4_1v87""","""australia_2026""",4,"""On_Track_Overtake""",1,87,"""P10 -> P7""","""MEDIUM""","""MEDIUM""",0.0,…,null,null,null,null,null,null,null,null,null,null
"""australia_2026_L4_1v5""","""australia_2026""",4,"""On_Track_Overtake""",1,5,"""P10 -> P7""","""MEDIUM""","""MEDIUM""",0.0,…,null,null,null,null,null,null,null,null,null,null
"""australia_2026_L4_3v10""","""australia_2026""",4,"""On_Track_Overtake""",3,10,"""P13 -> P9""","""HARD""","""UNKNOWN""",0.0,…,null,null,null,null,null,null,null,null,null,null
"""australia_2026_L4_3v5""","""australia_2026""",4,"""On_Track_Overtake""",3,5,"""P13 -> P9""","""HARD""","""MEDIUM""",0.0,…,null,null,null,null,null,null,null,null,null,null
